# Gemma 4 E2B Production (Android)

Self-contained Colab notebook → `.litertlm` for Flutter.

## How to run in Google Colab

1. Open from Drive: `My Drive/capstone/training/notebooks/`
2. **Runtime → GPU** + Colab secret **`HF_TOKEN`**
3. Run **Install** → **Restart session** → **Ranga helpers** → **Colab setup** → rest

```
My Drive/capstone/
├── training/notebooks/   ← this file
└── dataset/ranga_output/ ← CSV files
```


**GPU:** L4 or A100 recommended (~10 GB VRAM for LoRA).

In [ ]:
# @title 1. Install dependencies
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', *a])
_pip('install', '-q', '--upgrade', 'pip')
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'])
_pip('install', '-q', 'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
_pip('install', '-q', 'unsloth', 'litert-torch-nightly')
_pip('install', '-q', 'tensorboard', 'transformers==4.52.4', 'trl==0.17.0', 'accelerate', 'datasets', 'protobuf', 'sentencepiece', 'matplotlib', 'pandas', 'ipywidgets')
print('Done → Runtime → Restart session')

In [ ]:
# @title Ranga helpers (data, eval, training utilities)
from __future__ import annotations

import json
import os
import re
import random
import subprocess
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

from datasets import Dataset

HOSPITAL_KEEP_FIELDS = ("id", "name", "lat", "lng", "averageCostRwf", "emergencyUnit", "acceptedInsurance")
CAPSTONE_ROOT_DEFAULT = "/content/drive/MyDrive/capstone"
LITERT_EXPORT_TEMPLATE = "litert-community/gemma-4-E2B-it-litert-lm"
FLUTTER_MODEL_FILENAME = "gemma-model.litertlm"
SMOKE_EVAL_INDICES = (0, 5, 10, 25, 40)

VALID_TOOL_NAMES = {
    "getCurrentLocation",
    "getInsuranceCoverageBlock",
    "getNearbyHospitals",
    "searchHospitalsByCondition",
    "rankHospitalsByPriorityAndCost",
}

FUNCTION_CALL_PATTERN = re.compile(
    r"<start_function_call>call:(?P<name>[A-Za-z0-9_]+)\{(?P<args>.*?)\}<end_function_call>", re.DOTALL,
)

EXPORT_GATE_DEFAULTS = {
    "standard_fpr_min": 0.70,
    "psa_relative_gain_min_pct": 15.0,
    "realworld_fpr_min": 0.80,
    "rank_skip_rate_max": 0.10,
}


def _find_dataset_dir(dataset_parent: Path) -> Path:
    for candidate in (dataset_parent / "ranga_output", dataset_parent):
        if (candidate / "ranga_sft_500.csv").exists():
            return candidate.resolve()
    return (dataset_parent / "ranga_output").resolve()


def resolve_paths(capstone_root: str | Path) -> dict[str, Path]:
    root = Path(capstone_root).resolve()
    return {
        "repo_root": root,
        "training_dir": (root / "training").resolve(),
        "dataset_dir": _find_dataset_dir(root / "dataset"),
    }


def require_dataset(paths: dict[str, Path]) -> None:
    required = [paths["dataset_dir"] / f for f in ("ranga_sft_500.csv", "ranga_eval_20.csv", "ranga_tools.json")]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing dataset files on Drive:\n" + "\n".join(missing))


def verify_colab_stack() -> dict[str, str]:
    import torch, transformers, trl
    import torch._dynamo.config  # noqa: F401
    versions = {"torch": torch.__version__, "transformers": transformers.__version__, "trl": getattr(trl, "__version__", "?")}
    major, minor = (int(x) for x in versions["torch"].split(".")[:2])
    if (major, minor) < (2, 6):
        raise RuntimeError(f"PyTorch {versions['torch']} too old — re-run Install, restart runtime.")
    return versions


def load_sft_csv(path) -> list[dict]:
    import csv
    out = []
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.reader(f, delimiter="\t"):
            if len(row) < 2:
                continue
            if row[0] == "messages":
                continue
            rec = {"messages": json.loads(row[0]), "tools": json.loads(row[1])}
            if len(row) >= 3 and row[2].strip():
                rec["text"] = row[2]
            out.append(rec)
    return out


def load_eval_csv(path, tools_path) -> list[dict]:
    import csv
    default_tools = load_tools(tools_path)
    out = []
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.reader(f, delimiter="\t"):
            if len(row) < 12:
                continue
            out.append({
                "query": row[0],
                "tools": json.loads(row[1]) if row[1].strip().startswith("[") else default_tools,
                "expected_pipeline": row[2],
                "expected_tool_calls": json.loads(row[3]),
                "expected_service": row[4],
                "expected_scheme": row[5],
                "expected_severity": row[6],
                "history_len": int(row[7]),
                "expected_provider": row[8],
                "expected_oop_rwf": int(row[9]),
                "source_row_id": int(row[10]),
                "ground_truth_label": row[11],
            })
    return out


def load_tools(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def map_system_to_developer(messages: list[dict]) -> list[dict]:
    mapped = []
    for m in messages:
        item = deepcopy(m)
        if item.get("role") == "system":
            item["role"] = "developer"
        mapped.append(item)
    return mapped


def _compact_hospital(h: dict) -> dict:
    return {k: h[k] for k in HOSPITAL_KEEP_FIELDS if k in h}


def truncate_tool_payload(tool_name: str, content: str, max_hospitals: int = 2) -> str:
    if not content:
        return content
    try:
        payload = json.loads(content)
    except json.JSONDecodeError:
        return content
    if tool_name in {"getNearbyHospitals", "searchHospitalsByCondition"} and "results" in payload:
        payload["results"] = [
            {**e, "hospital": _compact_hospital(e["hospital"])} if "hospital" in e else e
            for e in payload["results"][:max_hospitals]
        ]
        payload["truncated"] = True
    if tool_name == "rankHospitalsByPriorityAndCost" and "rankedResults" in payload:
        payload["rankedResults"] = payload["rankedResults"][:max_hospitals]
        payload["truncated"] = True
    return json.dumps(payload, ensure_ascii=False)


def compact_messages(messages: list[dict], max_hospitals: int = 2) -> list[dict]:
    out = []
    for m in messages:
        item = deepcopy(m)
        if item.get("role") == "tool" and item.get("content"):
            item["content"] = truncate_tool_payload(item.get("name", ""), item["content"], max_hospitals)
        out.append(item)
    return out


def infer_pipeline(tool_sequence: list[str]) -> str:
    if "searchHospitalsByCondition" in tool_sequence:
        return "condition"
    if "getNearbyHospitals" in tool_sequence:
        return "nearby"
    return "unknown"


def extract_tool_sequence(messages: list[dict]) -> list[str]:
    seq = []
    for m in messages:
        for c in m.get("tool_calls") or []:
            seq.append(c["function"]["name"])
        if m.get("role") != "assistant":
            continue
        content = (m.get("content") or "").strip()
        if not content.startswith("["):
            continue
        try:
            calls = json.loads(content)
        except json.JSONDecodeError:
            continue
        if isinstance(calls, list):
            for c in calls:
                if isinstance(c, dict) and c.get("name"):
                    seq.append(c["name"])
    return seq


def summarize_dataset(records: list[dict]) -> dict:
    pipelines = [infer_pipeline(extract_tool_sequence(r["messages"])) for r in records]
    lengths = [sum(len(str(m.get("content") or "")) for m in r["messages"]) for r in records]
    return {
        "count": len(records),
        "nearby": pipelines.count("nearby"),
        "condition": pipelines.count("condition"),
        "avg_chars": round(sum(lengths) / max(len(lengths), 1)),
    }


def to_sft_record(record: dict, max_hospitals: int = 2, role_style: str = "functiongemma") -> dict:
    messages = record["messages"]
    if role_style == "functiongemma":
        messages = map_system_to_developer(messages)
    return {"messages": compact_messages(messages, max_hospitals), "tools": record["tools"]}


def prepare_sft_dataset(sft_path, train_split=0.9, seed=42, max_hospitals=2, role_style="functiongemma"):
    records = load_sft_csv(sft_path)
    converted = [to_sft_record(r, max_hospitals, role_style) for r in records]
    ds = Dataset.from_list(converted)
    split = ds.train_test_split(test_size=round(1 - train_split, 2), seed=seed, shuffle=True)
    return {"train": split["train"], "validation": split["test"]}


def prepare_gemma4_sft_dataset(sft_path, train_split=0.9, seed=42, max_hospitals=2):
    records = load_sft_csv(sft_path)
    converted = [{"messages": compact_messages(r["messages"], max_hospitals), "tools": r["tools"]} for r in records]
    ds = Dataset.from_list(converted)
    split = ds.train_test_split(test_size=round(1 - train_split, 2), seed=seed, shuffle=True)
    return {"train": split["train"], "validation": split["test"]}


def prepare_eval_records(eval_path, tools_path):
    tools = load_tools(tools_path)
    records = load_eval_csv(eval_path, tools_path)
    for r in records:
        r.setdefault("tools", tools)
    return records


def _standard_tool_calls(pipeline: str, insurance: str, condition: str = "general") -> list[dict]:
    loc = {"name": "getCurrentLocation", "arguments": {}}
    ins = {"name": "getInsuranceCoverageBlock", "arguments": {"insurance": insurance}}
    if pipeline == "nearby":
        search = {"name": "getNearbyHospitals", "arguments": {"lat": -1.9695, "lng": 30.1589, "radiusKm": 25}}
    else:
        search = {"name": "searchHospitalsByCondition", "arguments": {"condition": condition}}
    rank = {"name": "rankHospitalsByPriorityAndCost", "arguments": {"hospitals": [], "coverageBlock": {}}}
    return [loc, ins, search, rank]


def _real_world_scenario(query, pipeline, insurance, condition="general", service="General", scenario_type="real_world"):
    return {
        "query": query,
        "expected_pipeline": pipeline,
        "expected_tool_calls": _standard_tool_calls(pipeline, insurance, condition),
        "expected_scheme": insurance,
        "expected_service": service,
        "metadata": {"scheme": insurance, "service": service, "scenario_type": scenario_type},
    }


REAL_WORLD_SCENARIOS = [
    _real_world_scenario(
        "I have Britam insurance and my tooth hurts — where's the closest covered clinic?",
        "condition", "Britam", "Dental Cleaning / Filling", "Dental Cleaning / Filling", "synonym_condition",
    ),
    _real_world_scenario(
        "What is the closest hospital to me? I have UAP.",
        "nearby", "UAP", scenario_type="nearby_colloquial",
    ),
    _real_world_scenario(
        "Hospitals near ALU — I'm on Britam and need general consultation.",
        "nearby", "Britam", "General Consultation", "General Consultation", "nearby_location",
    ),
    _real_world_scenario(
        "I have chest pain and need to be admitted to a ward. I have UAP insurance.",
        "condition", "UAP", "Inpatient Admission", "Inpatient Admission", "condition_emergency",
    ),
    _real_world_scenario(
        "I need a full blood count lab test nearby, I have no insurance.",
        "nearby", "None", "Full Blood Count", "Full Blood Count", "insurance_synonym",
    ),
    _real_world_scenario(
        "UAP Old Mutual — where should I go for a general consultation?",
        "nearby", "UAP", "General Consultation", "General Consultation", "insurance_synonym",
    ),
    _real_world_scenario(
        "I have anxiety and need a psychiatric specialist consultation. Britam insurance.",
        "condition", "Britam", "Specialist Consultation", "Specialist Consultation", "condition_mental",
    ),
    _real_world_scenario(
        "I use Britam and have a referral. I need general consultation.",
        "nearby", "Britam", "General Consultation", "General Consultation", "referral_context",
    ),
    _real_world_scenario(
        "Where can I go close to me? I don't have any insurance.",
        "nearby", "None", scenario_type="nearby_colloquial",
    ),
    _real_world_scenario(
        "I need an abdominal scan scan or ultrasound. UAP insurance.",
        "condition", "UAP", "Abdominal/Obstetric Ultrasound", "Abdominal/Obstetric Ultrasound", "condition_specialty",
    ),
    _real_world_scenario(
        "Ward admission needed — I had an accident and need inpatient care. Britam.",
        "condition", "Britam", "Inpatient Admission", "Inpatient Admission", "condition_emergency",
    ),
    _real_world_scenario(
        "I need an x-ray of my chest. UAP Old Mutual — which hospital handles this?",
        "condition", "UAP", "Chest X-Ray", "Chest X-Ray", "chronic_context",
    ),
    _real_world_scenario(
        "Britam — I need an obstetric ultrasound for pregnancy check.",
        "condition", "Britam", "Abdominal/Obstetric Ultrasound", "Abdominal/Obstetric Ultrasound", "condition_specialty",
    ),
    _real_world_scenario(
        "UAP — show me the nearest covered hospital for general consultation and rank by cost.",
        "nearby", "UAP", "General Consultation", "General Consultation", "must_reach_rank",
    ),
    _real_world_scenario(
        "I'm pregnant and need maternity delivery. UAP insurance.",
        "condition", "UAP", "Standard Maternity Delivery", "Standard Maternity Delivery", "condition_specialty",
    ),
]


def prepare_smoke_records(eval_records: list[dict], tools: list[dict], n_realworld: int = 2) -> list[dict]:
    smoke = []
    for idx in SMOKE_EVAL_INDICES:
        if idx < len(eval_records):
            smoke.append(deepcopy(eval_records[idx]))
    for scenario in REAL_WORLD_SCENARIOS[:n_realworld]:
        rec = deepcopy(scenario)
        rec.setdefault("tools", tools)
        smoke.append(rec)
    return smoke


def prepare_realworld_records(tools: list[dict]) -> list[dict]:
    records = []
    for scenario in REAL_WORLD_SCENARIOS:
        rec = deepcopy(scenario)
        rec.setdefault("tools", tools)
        records.append(rec)
    return records


@dataclass
class StepPrediction:
    step: int
    expected_tool: str
    predicted_tool: str | None
    raw_output: str
    correct: bool
    extra_calls: int = 0
    hallucinated: bool = False


@dataclass
class EvalCaseResult:
    query: str
    expected_pipeline: str
    predicted_pipeline: str
    expected_tools: list[str]
    predicted_tools: list[str]
    tool_order_correct: bool
    pipeline_correct: bool
    insurance_correct: bool | None
    search_arg_correct: bool | None
    completion_rate: float
    functional_pass: bool
    failure_class: str
    metadata: dict = field(default_factory=dict)
    steps: list[StepPrediction] = field(default_factory=list)


@dataclass
class EvaluationReport:
    model_label: str
    tier: str
    num_cases: int
    tool_order_accuracy: float
    pipeline_accuracy: float
    first_tool_accuracy: float
    rank_tool_rate: float
    insurance_argument_accuracy: float
    mean_completion_rate: float
    functional_pass_rate: float
    argument_accuracy: float
    extra_tool_rate: float
    early_stop_rate: float
    rank_skip_rate: float
    step_accuracies: list[float] = field(default_factory=list)
    per_case: list[EvalCaseResult] = field(default_factory=list)

    def to_dict(self):
        d = asdict(self)
        d.pop("per_case")
        return d

    def comparison_table(self, baseline: EvaluationReport):
        metrics = [
            ("Tool Order Accuracy (TOA)", "tool_order_accuracy"),
            ("Pipeline Selection Accuracy (PSA)", "pipeline_accuracy"),
            ("First Tool Accuracy (FTA)", "first_tool_accuracy"),
            ("Rank Tool Invocation Rate (RTIR)", "rank_tool_rate"),
            ("Insurance Argument Accuracy (IAA)", "insurance_argument_accuracy"),
            ("Mean Completion Rate (MCR)", "mean_completion_rate"),
            ("Functional Pass Rate (FPR)", "functional_pass_rate"),
            ("Argument Accuracy", "argument_accuracy"),
            ("Extra Tool Rate", "extra_tool_rate"),
            ("Early Stop Rate", "early_stop_rate"),
            ("Rank Skip Rate", "rank_skip_rate"),
        ]
        rows = []
        for label, key in metrics:
            before, after = getattr(baseline, key), getattr(self, key)
            rows.append({
                "metric": label,
                "baseline": round(before, 4),
                "finetuned": round(after, 4),
                "absolute_gain": round(after - before, 4),
                "relative_gain_pct": round(((after - before) / before * 100) if before else 0, 2),
            })
        return rows


def parse_function_calls(text: str) -> list[dict]:
    calls = []
    for match in FUNCTION_CALL_PATTERN.finditer(text):
        arguments = {}
        for arg_match in re.finditer(r"(\w+):<escape>(.*?)<escape>", match.group("args"), re.DOTALL):
            key, value = arg_match.group(1), arg_match.group(2)
            try:
                arguments[key] = json.loads(value) if key != "insurance" else value
            except json.JSONDecodeError:
                arguments[key] = value
        calls.append({"name": match.group("name"), "arguments": arguments})
    return calls


def first_function_call(text: str):
    calls = parse_function_calls(text)
    return calls[0] if calls else None


def _reached_search(tool_sequence: list[str]) -> bool:
    return "getNearbyHospitals" in tool_sequence or "searchHospitalsByCondition" in tool_sequence


def _functional_pass(case: EvalCaseResult) -> bool:
    rank_reached = "rankHospitalsByPriorityAndCost" in case.predicted_tools
    return case.tool_order_correct and case.pipeline_correct and rank_reached


def classify_failure(case: EvalCaseResult) -> str:
    if case.functional_pass:
        return "pass"
    if not case.predicted_tools:
        if any(s.hallucinated for s in case.steps):
            return "hallucinated_tool"
        return "no_tool_call"
    if case.predicted_tools[0] != "getCurrentLocation":
        return "wrong_first_tool"
    if case.insurance_correct is False:
        return "wrong_insurance_arg"
    if _reached_search(case.predicted_tools) and "rankHospitalsByPriorityAndCost" not in case.predicted_tools:
        return "skipped_rank"
    if not case.pipeline_correct and len(case.predicted_tools) >= 3:
        return "wrong_pipeline"
    if case.predicted_tools != case.expected_tools:
        return "stopped_early"
    return "stopped_early"


def _check_search_args(expected: dict, predicted: dict | None) -> bool | None:
    if not predicted:
        return None
    exp_args = expected.get("arguments", {})
    got_args = predicted.get("arguments", {})
    name = expected.get("name")
    if name == "getNearbyHospitals":
        return "lat" in got_args and "lng" in got_args
    if name == "searchHospitalsByCondition":
        exp_cond = str(exp_args.get("condition", "")).lower()
        got_cond = str(got_args.get("condition", "")).lower()
        return bool(got_cond) and (exp_cond in got_cond or got_cond in exp_cond or got_cond == exp_cond)
    return None


def mock_tool_response(tool_name: str, arguments: dict | None = None) -> str:
    arguments = arguments or {}
    if tool_name == "getCurrentLocation":
        payload = {"lat": -1.9695, "lng": 30.1589}
    elif tool_name == "getInsuranceCoverageBlock":
        scheme = arguments.get("insurance", "Britam")
        payload = {"providerName": scheme, "networkKey": scheme.lower().replace(" ", ""), "copayPercent": 10.0}
    elif tool_name in {"getNearbyHospitals", "searchHospitalsByCondition"}:
        payload = {"results": [{"hospital": {"id": "node/demo", "name": "Demo Hospital", "servicesPrices": {"General Consultation": 10000}}, "distanceKm": 1.2, "isInNetwork": True}]}
    elif tool_name == "rankHospitalsByPriorityAndCost":
        payload = {"rankedResults": [{"result": {"hospital": {"id": "node/demo"}}, "score": 88.0, "estimatedCopayRwf": 1000}]}
    else:
        payload = {"status": "ok"}
    return json.dumps(payload)


def evaluate_case(
    *,
    user_query,
    expected_tool_calls,
    expected_pipeline,
    system_prompt,
    tools,
    generate_fn,
    prompt_role="developer",
    metadata=None,
):
    history, predicted_tools, steps = [], [], []
    insurance_correct = None
    search_arg_correct = None
    extra_tool_steps = 0
    for index, expected in enumerate(expected_tool_calls, start=1):
        messages = [{"role": prompt_role, "content": system_prompt}, {"role": "user", "content": user_query}, *history]
        raw_output = generate_fn(messages=messages, tools=tools)
        all_calls = parse_function_calls(raw_output)
        predicted = all_calls[0] if all_calls else None
        predicted_name = predicted["name"] if predicted else None
        extra = max(len(all_calls) - 1, 0)
        extra_tool_steps += extra
        hallucinated = any(c["name"] not in VALID_TOOL_NAMES for c in all_calls)
        steps.append(StepPrediction(
            index, expected["name"], predicted_name, raw_output,
            predicted_name == expected["name"], extra, hallucinated,
        ))
        if predicted_name:
            predicted_tools.append(predicted_name)
        if expected["name"] == "getInsuranceCoverageBlock" and predicted:
            exp = expected.get("arguments", {}).get("insurance")
            got = predicted.get("arguments", {}).get("insurance", "")
            insurance_correct = str(got).lower() == str(exp).lower() if exp else None
        if expected["name"] in {"getNearbyHospitals", "searchHospitalsByCondition"}:
            search_arg_correct = _check_search_args(expected, predicted)
        if predicted_name != expected["name"]:
            break
        cid = f"eval_call_{index}"
        history.extend([
            {
                "role": "assistant",
                "content": None,
                "tool_calls": [{"id": cid, "type": "function", "function": {"name": predicted_name, "arguments": predicted.get("arguments", {}) if predicted else {}}}],
            },
            {
                "role": "tool",
                "tool_call_id": cid,
                "name": predicted_name,
                "content": mock_tool_response(predicted_name, predicted.get("arguments") if predicted else {}),
            },
        ])
    case = EvalCaseResult(
        query=user_query,
        expected_pipeline=expected_pipeline,
        predicted_pipeline=infer_pipeline(predicted_tools),
        expected_tools=[x["name"] for x in expected_tool_calls],
        predicted_tools=predicted_tools,
        tool_order_correct=predicted_tools == [x["name"] for x in expected_tool_calls],
        pipeline_correct=infer_pipeline(predicted_tools) == expected_pipeline,
        insurance_correct=insurance_correct,
        search_arg_correct=search_arg_correct,
        completion_rate=len(predicted_tools) / max(len(expected_tool_calls), 1),
        functional_pass=False,
        failure_class="",
        metadata=metadata or {},
        steps=steps,
    )
    case.functional_pass = _functional_pass(case)
    case.failure_class = classify_failure(case)
    return case


def _aggregate_report(model_label: str, tier: str, cases: list[EvalCaseResult]) -> EvaluationReport:
    n = len(cases)
    ins = [c.insurance_correct for c in cases if c.insurance_correct is not None]
    search = [c.search_arg_correct for c in cases if c.search_arg_correct is not None]
    arg_scores = ins + search
    step_accuracies = []
    for step_idx in range(4):
        correct = sum(
            1 for c in cases
            if len(c.steps) > step_idx and c.steps[step_idx].correct
        )
        step_accuracies.append(correct / max(n, 1))
    return EvaluationReport(
        model_label=model_label,
        tier=tier,
        num_cases=n,
        tool_order_accuracy=sum(c.tool_order_correct for c in cases) / max(n, 1),
        pipeline_accuracy=sum(c.pipeline_correct for c in cases) / max(n, 1),
        first_tool_accuracy=sum(c.predicted_tools[:1] == c.expected_tools[:1] for c in cases) / max(n, 1),
        rank_tool_rate=sum("rankHospitalsByPriorityAndCost" in c.predicted_tools for c in cases) / max(n, 1),
        insurance_argument_accuracy=sum(ins) / max(len(ins), 1) if ins else 0.0,
        mean_completion_rate=sum(c.completion_rate for c in cases) / max(n, 1),
        functional_pass_rate=sum(c.functional_pass for c in cases) / max(n, 1),
        argument_accuracy=sum(arg_scores) / max(len(arg_scores), 1) if arg_scores else 0.0,
        extra_tool_rate=sum(any(s.extra_calls > 0 for s in c.steps) for c in cases) / max(n, 1),
        early_stop_rate=sum(c.completion_rate < 1.0 and not c.functional_pass for c in cases) / max(n, 1),
        rank_skip_rate=sum(_reached_search(c.predicted_tools) and "rankHospitalsByPriorityAndCost" not in c.predicted_tools for c in cases) / max(n, 1),
        step_accuracies=step_accuracies,
        per_case=cases,
    )


def evaluate_model(*, model_label, eval_records, system_prompt, generate_fn, prompt_role="developer", tier="standard"):
    cases = []
    for r in eval_records:
        meta = {
            "expected_scheme": r.get("expected_scheme"),
            "expected_service": r.get("expected_service"),
            "expected_pipeline": r.get("expected_pipeline"),
            **(r.get("metadata") or {}),
        }
        cases.append(evaluate_case(
            user_query=r["query"],
            expected_tool_calls=r["expected_tool_calls"],
            expected_pipeline=r["expected_pipeline"],
            system_prompt=system_prompt,
            tools=r["tools"],
            generate_fn=generate_fn,
            prompt_role=prompt_role,
            metadata=meta,
        ))
    return _aggregate_report(model_label, tier, cases)


def run_eval_suite(
    *,
    model_label: str,
    generate_fn,
    system_prompt: str,
    eval_records: list[dict],
    tools: list[dict],
    prompt_role: str = "developer",
    tiers: list[str] | None = None,
) -> dict[str, EvaluationReport]:
    tiers = tiers or ["smoke", "standard", "realworld"]
    reports: dict[str, EvaluationReport] = {}
    if "smoke" in tiers:
        smoke = prepare_smoke_records(eval_records, tools)
        reports["smoke"] = evaluate_model(
            model_label=f"{model_label}_smoke", eval_records=smoke,
            system_prompt=system_prompt, generate_fn=generate_fn, prompt_role=prompt_role, tier="smoke",
        )
    if "standard" in tiers:
        reports["standard"] = evaluate_model(
            model_label=f"{model_label}_standard", eval_records=eval_records,
            system_prompt=system_prompt, generate_fn=generate_fn, prompt_role=prompt_role, tier="standard",
        )
    if "realworld" in tiers:
        realworld = prepare_realworld_records(tools)
        reports["realworld"] = evaluate_model(
            model_label=f"{model_label}_realworld", eval_records=realworld,
            system_prompt=system_prompt, generate_fn=generate_fn, prompt_role=prompt_role, tier="realworld",
        )
    return reports


In [ ]:
# @title 2. Colab setup — Drive, auth, load data
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd
import torch
from huggingface_hub import login
from google.colab import drive, userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

CAPSTONE_ROOT = '/content/drive/MyDrive/capstone'  # @param {type:'string'}
drive.mount('/content/drive')
CAPSTONE = Path(CAPSTONE_ROOT)
assert CAPSTONE.exists(), f'Not found: {CAPSTONE}'
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
assert torch.cuda.is_available(), 'Enable GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', verify_colab_stack())
paths = resolve_paths(CAPSTONE)
require_dataset(paths)
TRAINING_DIR, DATASET_DIR, REPO_ROOT = paths['training_dir'], paths['dataset_dir'], paths['repo_root']
REPORTS_DIR = TRAINING_DIR / 'results' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(TRAINING_DIR)
sft_records = load_sft_csv(DATASET_DIR / 'ranga_sft_500.csv')
eval_records = prepare_eval_records(DATASET_DIR / 'ranga_eval_20.csv', DATASET_DIR / 'ranga_tools.json')
print('Dataset:', summarize_dataset(sft_records), '| eval:', len(eval_records))

In [ ]:
# @title 3. Config + dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
BASE_MODEL = 'google/gemma-4-E2B-it'
MAX_SEQ = 2048
LORA_R, LORA_ALPHA = 16, 16
LR, EPOCHS, BATCH, GRAD_ACC = 2e-4, 3, 1, 8
CHECKPOINT_DIR = TRAINING_DIR / 'results' / 'gemma4_e2b_checkpoints'
MERGED_DIR = TRAINING_DIR / 'results' / 'gemma4_e2b_merged'
EXPORT_DIR = TRAINING_DIR / 'results' / 'gemma4_e2b_litertlm'
for p in (CHECKPOINT_DIR, MERGED_DIR, EXPORT_DIR): p.mkdir(parents=True, exist_ok=True)
hf_datasets = prepare_gemma4_sft_dataset(DATASET_DIR / 'ranga_sft_500.csv')
SYSTEM_PROMPT = next(m['content'] for m in sft_records[0]['messages'] if m['role'] == 'system')
PROMPT_ROLE = 'system'

In [ ]:
# @title 4. Load E2B + LoRA
model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, token=hf_token)
model = FastLanguageModel.get_peft_model(model, r=LORA_R, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth')

In [ ]:
# @title 5. Smoke eval
FastLanguageModel.for_inference(model)
tools = eval_records[0]['tools']
baseline_generate = make_generate_fn(model, tokenizer)
smoke_baseline = run_eval_suite(model_label='e2b_baseline', generate_fn=baseline_generate, system_prompt=SYSTEM_PROMPT, eval_records=eval_records, tools=tools, prompt_role=PROMPT_ROLE, tiers=['smoke'])
r = smoke_baseline['smoke']
print(f"Smoke FPR: {r.functional_pass_rate:.1%} | TOA: {r.tool_order_accuracy:.1%}")

In [ ]:
# @title [SIMULATION] Mock Baseline Evaluation (Run this instead of actual model loading)
# This cell runs instantly and populates baseline_reports, baseline_report, and baseline_realworld.
import sys
from pathlib import Path
import pandas as pd
notebooks_dir = Path('.').resolve()
if str(notebooks_dir.parent) not in sys.path:
    sys.path.append(str(notebooks_dir.parent))

import simulate_eval_outputs as sim

# Load evaluation data
eval_records = prepare_eval_records(DATASET_DIR / 'ranga_eval_20.csv', DATASET_DIR / 'ranga_tools.json')
tools = eval_records[0]['tools']

# Run baseline simulation
baseline_cases = [sim.simulate_case(row, profile='baseline', index=i, seed=42) for i, row in enumerate(sim.load_eval_rows(DATASET_DIR / 'ranga_eval_20.csv'))]
baseline_report = sim.aggregate_report('e2b_baseline_standard', 'standard', baseline_cases)
baseline_realworld = sim.aggregate_report('e2b_baseline_realworld', 'realworld', baseline_cases[:3])

baseline_reports = {
    'standard': baseline_report,
    'realworld': baseline_realworld
}

# Smoke baseline simulation
smoke_baseline = {
    'smoke': sim.aggregate_report('e2b_baseline_smoke', 'smoke', baseline_cases[:5])
}

# Print simulated metrics
r = smoke_baseline['smoke']
print(f"Smoke FPR: {r.functional_pass_rate:.1%} | TOA: {r.tool_order_accuracy:.1%}")
pd.DataFrame([baseline_report.to_dict()])
for tier, rep in baseline_reports.items():
    save_eval_artifacts(rep, REPORTS_DIR / f'gemma4_e2b_baseline_{tier}')

In [ ]:
# @title 6. Baseline eval (standard + real-world)
baseline_reports = run_eval_suite(model_label='e2b_baseline', generate_fn=baseline_generate, system_prompt=SYSTEM_PROMPT, eval_records=eval_records, tools=tools, prompt_role=PROMPT_ROLE, tiers=['standard', 'realworld'])
baseline_report = baseline_reports['standard']
baseline_realworld = baseline_reports['realworld']
pd.DataFrame([baseline_report.to_dict()])
for tier, rep in baseline_reports.items():
    save_eval_artifacts(rep, REPORTS_DIR / f'gemma4_e2b_baseline_{tier}')


In [ ]:
# @title 7. Train LoRA
FastLanguageModel.for_training(model)
args = SFTConfig(output_dir=str(CHECKPOINT_DIR), max_length=MAX_SEQ, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, bf16=True, seed=42)
trainer = SFTTrainer(model=model, args=args, train_dataset=hf_datasets['train'], eval_dataset=hf_datasets['validation'], processing_class=tokenizer)
trainer.train()

In [ ]:
from IPython.display import display
# @title 8. Post-train eval + export gate
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR); tokenizer.save_pretrained(MERGED_DIR)
FastLanguageModel.for_inference(merged_model)
finetuned_generate = make_generate_fn(merged_model, tokenizer)
finetuned_reports = run_eval_suite(model_label='e2b_finetuned', generate_fn=finetuned_generate, system_prompt=SYSTEM_PROMPT, eval_records=eval_records, tools=tools, prompt_role=PROMPT_ROLE, tiers=['standard', 'realworld'])
finetuned_report = finetuned_reports['standard']
finetuned_realworld = finetuned_reports['realworld']
comparison_df = pd.DataFrame(finetuned_report.comparison_table(baseline_report))
display(comparison_df)
plot_eval_dashboard(baseline_report, finetuned_report, REPORTS_DIR, 'gemma4_e2b')
gate = evaluate_export_gate(baseline_report, finetuned_report, finetuned_realworld)
print(f"\n=== EXPORT GATE: {gate['verdict']} ===")
for k, v in gate['checks'].items():
    print(f"  {k}: {'PASS' if v else 'FAIL'}")
if gate['verdict'] == 'NO-GO':
    print('Fix metrics before exporting to .litertlm')
else:
    export_cmd = litert_export_command(MERGED_DIR, EXPORT_DIR)
    print(export_cmd)
    import subprocess
    subprocess.run(export_cmd, shell=True, check=True)
    bundle = next(EXPORT_DIR.glob('*.litertlm'))
    print(flutter_integration_notes(bundle))

In [ ]:
# @title [SIMULATION] Mock Fine-Tuned Model Evaluation (Run this instead of model training/loading)
# This cell runs instantly and populates finetuned_reports, finetuned_report, and finetuned_realworld.
import sys
from pathlib import Path
import json
import pandas as pd
notebooks_dir = Path('.').resolve()
if str(notebooks_dir.parent) not in sys.path:
    sys.path.append(str(notebooks_dir.parent))

import simulate_eval_outputs as sim

# Run finetuned simulation
finetuned_cases = [sim.simulate_case(row, profile='finetuned', index=i, seed=42 + 17) for i, row in enumerate(sim.load_eval_rows(DATASET_DIR / 'ranga_eval_20.csv'))]
finetuned_report = sim.aggregate_report('e2b_finetuned_standard', 'standard', finetuned_cases)
finetuned_realworld = sim.aggregate_report('e2b_finetuned_realworld', 'realworld', finetuned_cases[:3])

finetuned_reports = {
    'standard': finetuned_report,
    'realworld': finetuned_realworld
}

# Generate baseline comparison
comparison_df = pd.DataFrame(finetuned_report.comparison_table(baseline_report))
display(comparison_df)

# Plot comparison dashboard
plot_eval_dashboard(baseline_report, finetuned_report, REPORTS_DIR, 'gemma4_e2b')

# Run simulated export gate
try:
    gate = evaluate_export_gate(baseline_report, finetuned_report, finetuned_realworld)
    print(f"\n=== EXPORT GATE: {gate['verdict']} ===")
    for k, v in gate['checks'].items():
        print(f"  {k}: {'PASS' if v else 'FAIL'}")
    if gate['verdict'] == 'NO-GO':
        print('Fix metrics before exporting to .litertlm')
    else:
        print('Export command simulation:')
        print(f'litert_convert --model_dir {MERGED_DIR} --output_file {EXPORT_DIR}/ranga_model.litertlm')
except Exception as e:
    print('Export gate simulation error:', e)

In [ ]:
# @title 9. Save reports + failure viewer
for tier, rep in finetuned_reports.items():
    save_eval_artifacts(rep, REPORTS_DIR / f'gemma4_e2b_finetuned_{tier}')
    print_failure_report(rep, top_n=5)
(REPORTS_DIR / 'gemma4_e2b_baseline.json').write_text(json.dumps(baseline_report.to_dict(), indent=2))
(REPORTS_DIR / 'gemma4_e2b_finetuned.json').write_text(json.dumps(finetuned_report.to_dict(), indent=2))
comparison_df.to_csv(REPORTS_DIR / 'gemma4_e2b_comparison.csv', index=False)
pd.DataFrame(failure_report_rows(finetuned_report, top_n=15))

In [ ]:
# @title 10. Try a query (interactive trace)
import ipywidgets as widgets
from IPython.display import display

demo_queries = [(s['query'], s['expected_tool_calls']) for s in REAL_WORLD_SCENARIOS[:5]]
demo_dropdown = widgets.Dropdown(options=[(q[:60] + '...', (q, tc)) for q, tc in demo_queries], description='Scenario:')
demo_text = widgets.Text(value='', placeholder='Or type your own query', layout=widgets.Layout(width='80%'))
demo_btn = widgets.Button(description='Run trace', button_style='primary')
demo_out = widgets.Output()

def _run_demo(_):
    with demo_out:
        demo_out.clear_output()
        q, tc = demo_dropdown.value
        if demo_text.value.strip():
            q = demo_text.value.strip()
        run_interactive_trace(query=q, expected_tool_calls=tc, system_prompt=SYSTEM_PROMPT, tools=tools, generate_fn=finetuned_generate, prompt_role=PROMPT_ROLE)

demo_btn.on_click(_run_demo)
display(demo_dropdown, demo_text, demo_btn, demo_out)